# 07: Offline Evaluation

This notebook walks through the **Phase 2 offline evaluation suite** for the
Knowledge-Grounded QA Agent. Each evaluator runs *after* the agent has responded,
using a Langfuse dataset as the source of truth.

## What You'll Learn

0. **Browse questions** and add per-question ground truth annotations
1. How to run the agent and capture the full `AgentResponse` (plan, tool calls, sources)
2. **Replanning Rate** — deterministic counter (no LLM needed)
3. **Plan Quality** — LLM judge on the research plan
4. **Tool Selection & Efficiency** — LLM judge on the tool call sequence
5. **Source Validation** — LLM judge on source authority and relevance
6. **Knowledge Base Usage** — checks vertex_search was called when required
7. Running the full experiment against the ground truth dataset from Notebook 06

## Prerequisites

Complete Notebooks 01–06 (especially Notebook 06 — ground truth must be uploaded
to Langfuse before running the full experiment). All credentials in `.env`:
- `GOOGLE_API_KEY`
- `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` (for LLM judges)
- `VERTEX_AI_DATASTORE_ID` (optional — only for KB evaluation)

In [ ]:
import json
import os
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.knowledge_qa import KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.evaluation.graders.plan_quality import (
    derive_plan_rubric,
    evaluate_plan_quality,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.replanning import (
    evaluate_replanning_rate,
    get_max_replan_threshold,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.source_validation import (
    evaluate_source_validation,
    get_source_rubric,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.tool_selection import (
    derive_tool_pattern,
    evaluate_tool_selection,
)
from aieng.agent_evals.knowledge_qa.evaluation.offline import (
    KnowledgeQATask,
    deepsearchqa_evaluator,
    knowledge_base_evaluator,
    plan_quality_evaluator,
    replanning_evaluator,
    source_validation_evaluator,
    tool_selection_evaluator,
)
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from dotenv import load_dotenv
from IPython.display import HTML, display  # noqa: A004
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

# All evaluation runs against this 9-example ground truth dataset
DATASET_NAME = "KnowledgeQA-GroundTruth"

## 0. Browse Questions and Add Ground Truth Annotations

The LLM judges (Plan Quality, Tool Selection) are more accurate when they have
per-question ground truth to anchor on — specifically:

| Field | Evaluator | Why it matters |
|---|---|---|
| `plan_rubric.must_cover` | Plan Quality | Tells the judge which concepts the plan must address |
| `tool_pattern.reference_tool_call_count` | Tool Selection | Tells the judge how many calls an expert would make |

Run the cell below to print all questions with their IDs, then fill in
`data/ground_truth_annotations.jsonl` for the ones you want to annotate.
All other fields are auto-derived — you only need to add what you know.

### How to add annotations

After identifying a question above, open `data/ground_truth_annotations.jsonl`
and add one line per question in this format:

```json
{"example_id": 1234, "plan_rubric": {"must_cover": ["GDP growth rate", "Statistics Canada source", "2023 timeframe"]}, "tool_pattern": {"reference_tool_call_count": 3}}
```

Only add the fields you want to override — everything else is auto-derived.
Then re-run the cell below to verify the annotation was loaded.

## 1. Run the Agent

All offline evaluators take their inputs from the `AgentResponse` object.
We run the agent once here and use the result across all subsequent sections.

In [ ]:
from types import SimpleNamespace

gt_path = Path("implementations/knowledge_qa/data/ground_truth_dataset.jsonl")
gt_examples = []
with open(gt_path) as f:
    for line in f:
        record = json.loads(line.strip())
        meta = record.get("metadata", {})
        gt_examples.append(SimpleNamespace(
            example_id=meta.get("example_id"),
            problem_category=meta.get("category", ""),
            answer_type=meta.get("answer_type", ""),
            problem=record["input"],
            answer=record["expected_output"],
            metadata=meta,
        ))

# Change index to run a different example (0–8)
example = gt_examples[0]

console.print(
    Panel(
        f"[bold]ID:[/bold] {example.example_id}\n"
        f"[bold]Category:[/bold] {example.problem_category}\n"
        f"[bold]Answer Type:[/bold] {example.answer_type}\n\n"
        f"[bold cyan]Question:[/bold cyan]\n{example.problem}\n\n"
        f"[bold yellow]Ground Truth:[/bold yellow]\n{example.answer}",
        title="Evaluation Example",
        border_style="blue",
    )
)

agent = KnowledgeGroundedAgent(enable_planning=True)
response = await run_with_display(agent, example.problem)

display_response(
    console,
    response.text,
    title="Agent Answer",
    subtitle=f"Duration: {response.total_duration_ms / 1000:.1f}s  |  Tools: {len(response.tool_calls)}  |  Replan count: {response.replan_count}",
)

## 2. Replanning Rate

Fully deterministic — no annotation needed. Reads `replan_count` directly from
`AgentResponse` and compares against the auto-derived threshold.

| Score | Meaning |
|---|---|
| `Replanning/Count` | Raw number of replannings |
| `Replanning/Flag` | 1 if count exceeds the threshold for this question type |
| `Replanning/Ratio` | replan_count / plan_steps |

In [ ]:
threshold = get_max_replan_threshold(
    answer_type=example.answer_type,
    problem_category=example.problem_category,
)
plan_steps = len(response.plan.steps) if response.plan else 1

replan_evals = evaluate_replanning_rate(
    replan_count=response.replan_count,
    plan_steps=plan_steps,
    max_replan_threshold=threshold,
)

t = Table(title="Replanning Rate")
t.add_column("Metric", style="cyan")
t.add_column("Value", style="white")
t.add_column("Note", style="dim")
for ev in replan_evals:
    note = f"threshold={threshold}" if ev.name == "Replanning/Flag" else ""
    t.add_row(ev.name, str(ev.value), note)
console.print(t)

## 3. Plan Quality

LLM judge on five dimensions. The rubric is auto-derived, but if you added
`must_cover` in the annotations file for this `example_id`, it will be merged
in automatically — giving the judge a concrete checklist to work from.

In [ ]:
plan_rubric = derive_plan_rubric(
    answer_type=example.answer_type,
    problem_category=example.problem_category,
)
console.print("[dim]Plan rubric:[/dim]", plan_rubric)

plan_descriptions = [
    step.description for step in response.plan.steps
] if response.plan else []

plan_evals = await evaluate_plan_quality(
    question=example.problem,
    plan_steps=plan_descriptions,
    plan_rubric=plan_rubric,
)

t = Table(title="Plan Quality")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in plan_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 4. Tool Selection & Efficiency

If you annotated `reference_tool_call_count` for this question, the judge uses
that instead of the category average for scoring `Efficiency/CallVolume`.

In [ ]:
tool_pattern = derive_tool_pattern(
    problem_category=example.problem_category,
    answer_type=example.answer_type,
)
console.print("[dim]Tool pattern:[/dim]", tool_pattern)

seq_table = Table(title=f"Tool Calls ({len(response.tool_calls)} total)")
seq_table.add_column("#", style="dim", justify="right")
seq_table.add_column("Tool", style="cyan")
seq_table.add_column("Args (preview)", style="white")
for i, tc in enumerate(response.tool_calls, 1):
    args_str = str(tc.get("args", {}))
    seq_table.add_row(str(i), tc.get("name", "?"), args_str[:60])
console.print(seq_table)

tool_evals = await evaluate_tool_selection(
    question=example.problem,
    tool_calls=response.tool_calls,
    final_answer=response.text,
    tool_pattern=tool_pattern,
)

t = Table(title="Tool Selection & Efficiency")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in tool_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 5. Source Validation

Fully auto-derived from `CATEGORY_SOURCE_RUBRIC` — no annotation needed.
vertex_search sources are skipped (already grounded in the private KB).

In [ ]:
source_rubric = get_source_rubric(example.problem_category)
console.print("[dim]Source rubric:[/dim]", source_rubric)

sources_dict = [{"title": s.title, "uri": s.uri} for s in response.sources]
console.print(f"[dim]{len(sources_dict)} sources cited[/dim]")

source_evals = await evaluate_source_validation(
    question=example.problem,
    problem_category=example.problem_category,
    sources=sources_dict,
    source_rubric=source_rubric,
)

t = Table(title="Source Validation")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in source_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 6. Final Output Quality (DeepSearchQA)

Compares the agent's final answer against the ground truth `expected_output` using
the DeepSearchQA grader — the same grader used in the full experiment.

| Score | Meaning |
|---|---|
| `DeepSearchQA/F1` | Harmonic mean of precision and recall |
| `DeepSearchQA/Precision` | Fraction of agent answer tokens that appear in ground truth |
| `DeepSearchQA/Recall` | Fraction of ground truth tokens covered by agent answer |
| `DeepSearchQA/Outcome` | Pass / Partial / Fail label |

In [ ]:
output_evals = await deepsearchqa_evaluator(
    input=example.problem,
    output={"text": response.text},
    expected_output=example.answer,
    metadata=example.metadata,
)

t = Table(title="Final Output Quality")
t.add_column("Metric", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in output_evals:
    score = f"{ev.value:.3f}" if isinstance(ev.value, float) else str(ev.value)
    t.add_row(ev.name, score, ev.comment or "")
console.print(t)

## 7. Full Experiment: All Evaluators via `run_experiment`

Runs against the `KnowledgeQA-GroundTruth` dataset uploaded in Notebook 06.
Each item already has `must_cover`, `reference_tool_call_count`, `source_rubric`,
and `requires_knowledge_base` baked into its metadata — the judges receive
concrete ground truth instead of auto-derived defaults.

> One agent call + six evaluator calls per dataset item.
> With 9 items and `max_concurrency=2`, expect ~15–30 minutes.

In [ ]:
task = KnowledgeQATask()

experiment_result = run_experiment(
    DATASET_NAME,
    name="knowledge-agent-full-offline-eval",
    description="All Phase 2 offline evaluators: F1, replanning, plan quality, tool selection, source validation",
    task=task.run,
    evaluators=[
        deepsearchqa_evaluator,
        replanning_evaluator,
        plan_quality_evaluator,
        tool_selection_evaluator,
        knowledge_base_evaluator,
        source_validation_evaluator,
    ],
    max_concurrency=2,
)

console.print("[green]Experiment complete[/green]")
if hasattr(experiment_result, "dataset_run_url") and experiment_result.dataset_run_url:
    display(HTML(
        f'<p>View in Langfuse: <a href="{experiment_result.dataset_run_url}" target="_blank">'
        f'{experiment_result.dataset_run_url}</a></p>'
    ))

## 8. Inspecting Results

In [ ]:
rows = []
for item_result in experiment_result.item_results:
    item = item_result.item
    question = str(item.input)
    row = {"question": question[:50] + "..." if len(question) > 50 else question}
    for ev in item_result.evaluations or []:
        row[ev.name] = ev.value
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

numeric_metrics = [c for c in df.columns if c != "question"]
if numeric_metrics:
    means = Table(title="Mean Scores Across Dataset")
    means.add_column("Metric", style="cyan")
    means.add_column("Mean", style="white", justify="right")
    for col in sorted(numeric_metrics):
        if df[col].dtype in ("float64", "int64"):
            means.add_row(col, f"{df[col].mean():.3f}")
    console.print(means)

## 9. Correlation Analysis: F1 vs Plan Quality and Source Quality

Compares the agent's output quality (F1) against its process quality scores to
identify whether planning or source selection is the primary driver of answer accuracy.

| Correlation (r) | Signal |
|---|---|
| ≥ 0.7 | Strong — that dimension is a reliable predictor of output quality |
| 0.4 – 0.7 | Moderate — worth investigating further |
| < 0.4 | Weak / inconclusive — likely other factors at play |

> With 9 examples, treat results as directional signal only.

In [ ]:
from aieng.agent_evals.async_client_manager import AsyncClientManager

f1_col = "DeepSearchQA/F1"
plan_cols = [c for c in df.columns if c.startswith("PlanQuality/")]
source_cols = [c for c in df.columns if c.startswith("SourceValidation/")]

if f1_col not in df.columns:
    console.print("[yellow]DeepSearchQA/F1 not found — run the full experiment first.[/yellow]")
else:
    df_corr = df[["question", f1_col]].copy()
    if plan_cols:
        df_corr["Plan Quality"] = df[plan_cols].mean(axis=1)
    if source_cols:
        df_corr["Source Quality"] = df[source_cols].mean(axis=1)

    score_cols = [c for c in ["Plan Quality", "Source Quality"] if c in df_corr.columns]

    # Per-example side-by-side table
    t = Table(title="F1 vs Quality Scores — Per Example")
    t.add_column("Question", style="white", width=38)
    t.add_column("F1", style="cyan", justify="right")
    for col in score_cols:
        t.add_column(col, style="yellow", justify="right")
    for _, row in df_corr.iterrows():
        t.add_row(
            row["question"],
            f"{row[f1_col]:.3f}",
            *[f"{row[c]:.2f}" for c in score_cols],
        )
    console.print(t)

    # Compute correlations
    def _interp(r: float) -> str:
        if abs(r) >= 0.7:
            return "strong signal"
        if abs(r) >= 0.4:
            return "moderate signal"
        return "weak / inconclusive"

    correlations: dict[str, float] = {}
    corr_t = Table(title="Pearson Correlation with F1")
    corr_t.add_column("Comparison", style="cyan")
    corr_t.add_column("r", style="white", justify="right")
    corr_t.add_column("Signal", style="dim")
    for col in score_cols:
        valid = df_corr[[f1_col, col]].dropna()
        if len(valid) >= 3:
            r = float(valid[f1_col].corr(valid[col]))
            correlations[col] = r
            corr_t.add_row(f"F1 vs {col}", f"{r:.3f}", _interp(r))
        else:
            corr_t.add_row(f"F1 vs {col}", "—", "not enough data")
    console.print(corr_t)
    console.print("[dim]9 examples — treat correlation as directional, not conclusive.[/dim]")

    # Push correlation scores to Langfuse as a summary trace
    if correlations:
        client_manager = AsyncClientManager.get_instance()
        langfuse_client = client_manager.langfuse_client
        summary_trace = langfuse_client.trace(
            name="KnowledgeQA/CorrelationSummary",
            metadata={"dataset": DATASET_NAME, "experiment": "knowledge-agent-full-offline-eval"},
        )
        for col, r in correlations.items():
            metric_name = col.replace(" ", "")
            langfuse_client.create_score(
                trace_id=summary_trace.id,
                name=f"Correlation/F1_vs_{metric_name}",
                value=round(r, 3),
                comment=_interp(r),
            )
        langfuse_client.flush()
        console.print(f"[green]✓[/green] Correlation scores pushed to Langfuse trace [bold]KnowledgeQA/CorrelationSummary[/bold]")

## Summary

In this notebook you:

0. **Browsed questions** by category, found their `example_id` values, and added
   `must_cover` / `reference_tool_call_count` annotations to `ground_truth_annotations.jsonl`
1. **Ran** the agent and captured the full `AgentResponse`
2. **Evaluated replanning rate** deterministically (no annotation needed)
3. **Evaluated plan quality** with annotations merged in for `must_cover`
4. **Evaluated tool selection** with `reference_tool_call_count` from annotations
5. **Evaluated source quality** fully auto-derived
6. **Ran the full experiment** with all evaluators — annotations applied automatically
7. **Inspected results** programmatically and in Langfuse

### Annotation workflow going forward

1. Run Section 0 to browse questions and note their `example_id` values
2. Add entries to `data/ground_truth_annotations.jsonl`
3. Re-run `python data/langfuse_upload.py` — annotations are merged into the Langfuse metadata
4. Re-run the experiment — annotated items get richer rubrics automatically